# CS 412 Machine Learning
## Spring 2026
## Homework 4

---

**Due:** April 27, 11:55 pm  
**Late Accepted Until:** April 29, 11:55 pm

### Student Information

In [ ]:
NAME       = "Your Name Lastname"     # e.g. "Emine Ayşe Sunar"
STUDENT_ID = "Your Student ID"        # e.g. "12345"

print(f"Student : {NAME}")
print(f"ID      : {STUDENT_ID}")

## Instructions

- **Run all cells before submitting.** All outputs must be visible. The notebook will not be re-run during grading. Cells with no output will receive zero points.
- **Submission:** Upload your notebook as `CS412-HW4-FirstnameLastname.ipynb`.
- **Late policy:** Up to 2 days late accepted with a 10-point penalty per day. Submissions within the first hour after the deadline incur only a 5-point penalty.

---

## Overview

In this assignment, you will build and compare a **Multilayer Perceptron (MLP)** and a **Convolutional Neural Network (CNN)** for image classification on a 10-class subset of **Food-101**, a benchmark dataset of 101,000 food images across 101 categories (pizza, sushi, waffles, etc.), originally published at ECCV 2014. We work with 10 classes to keep training times manageable on Colab. The dataset is available on Sucourse as `Food10.zip`.

**Minimum test accuracy required:**
- MLP: **30%**
- CNN: **50%**

> ⚠️ **Important:** Reaching the minimum accuracy threshold is required. If your model does not meet the threshold, you will receive at most half of the points for that part, even if the rest of your implementation is correct.

---

## Grading

- **Part 1**: Numerical Questions: 20 pts
- **Part 2**: MLP Classifier: 25 pts
- **Part 3**: CNN Classifier: 40 pts
- **Part 4**: Comparison & Analysis: 15 pts
- **Total: 100 pts**

## Part 0: Setup and Data Loading

> 📌 **Note:** This section contains setup and data loading code. Run all cells in this section without modifying them.

Run the cell below to install and import all required libraries. Make sure you are using a GPU runtime on Colab (Runtime > Change runtime type > T4 GPU).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import random

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### 0.1 Data Loading

Download `Food10.zip` from Sucourse and upload it to your Google Drive **without renaming it**. Then run the cells below to mount your Drive and extract the dataset.

The dataset is a 10-class subset of [Food-101](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/) (Bossard et al., ECCV 2014). It contains 750 training and 250 test images per class across 10 food categories: pizza, sushi, waffles, ice cream, chocolate cake, hamburger, hot dog, ramen, donuts, and fried rice.

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os

zip_path  = "/content/drive/My Drive/Food10.zip"
target_dir = "/content/data"

if not os.path.exists(os.path.join(target_dir, "train")):
    print("Extracting dataset...")
    !unzip -q "{zip_path}" -d "/content/data"
    print("Extraction complete.")
else:
    print("Dataset already extracted, skipping.")

data_dir = target_dir
print("Using dataset at:", data_dir)

### 0.2 Preprocessing and Visualization

We resize all images to 64×64, apply normalization, and split the training data into train (80%) and validation (20%) sets.

In [ ]:
IMG_SIZE = 64
SELECTED_CLASSES = [
    "pizza", "sushi", "waffles", "ice_cream", "chocolate_cake",
    "hamburger", "hot_dog", "ramen", "donuts", "fried_rice"
]

class FoodDataset(Dataset):
    def __init__(self, root_dir, split, transform=None):
        self.transform = transform
        self.samples = []
        split_dir = os.path.join(root_dir, split)
        for label, class_name in enumerate(SELECTED_CLASSES):
            class_dir = os.path.join(split_dir, class_name)
            for fname in os.listdir(class_dir):
                if fname.endswith(".jpg"):
                    self.samples.append((os.path.join(class_dir, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# Compute mean and std from training set only
def compute_mean_std(dataset):
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
    mean = torch.zeros(3)
    std = torch.zeros(3)
    for images, _ in loader:
        for c in range(3):
            mean[c] += images[:, c, :, :].mean()
            std[c] += images[:, c, :, :].std()
    mean /= len(loader)
    std /= len(loader)
    return mean, std

pre_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])
temp_dataset = FoodDataset(data_dir, "train", transform=pre_transform)
mean, std = compute_mean_std(temp_dataset)
print(f"Mean: {mean}")
print(f"Std:  {std}")

# Define transforms using computed stats
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Create datasets and loaders
full_train_dataset = FoodDataset(data_dir, "train", transform=transform_train)
test_dataset       = FoodDataset(data_dir, "test",  transform=transform_eval)

val_size   = int(0.2 * len(full_train_dataset))
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size],
                                          generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# Visualize one sample per class
display_dataset = FoodDataset(data_dir, "train", transform=transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
]))

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
shown = {}
idx = 0
while len(shown) < 10:
    img, label = display_dataset[idx]
    if label not in shown:
        shown[label] = img
    idx += 1

for ax, (label, img) in zip(axes.flatten(), sorted(shown.items())):
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(SELECTED_CLASSES[label], fontsize=9, pad=4)
    ax.axis("off")

plt.suptitle("One sample per class", fontsize=12)
plt.tight_layout(pad=2.5)
plt.show()

## Part 1: Numerical Questions [20 pts]

> 📌 **Note:** The numerical questions in this section are independent of the Food-101 dataset. The input dimensions and configurations given here are hypothetical and are not related to the Food-101 dataset.

Answer the following questions in the answer cells provided. No code is needed for this part.

### Question 1.1 — Output Size [8 pts]

You have an input feature map of spatial size **40×40** with **3 channels**. You apply a convolutional layer with the following configuration:
- Filter size: 5×5
- Number of filters: 16
- Stride: 2
- Padding: 1

**(a)** What is the spatial size (height × width) of the output feature map? Show your calculation using the formula $W_{out} = \lfloor \frac{W + 2P - F}{S} \rfloor + 1$.

**(b)** What is the total number of learnable parameters in this layer, including biases?

**(c)** What would the output spatial size be if you instead used stride 1 and padding 2? Show your work. What is this special case of padding called, and what is its purpose?

**Your answer for 1.1:**

(a)

(b)

(c)

### Question 1.2 — Conceptual Questions [12 pts]

**(a)** What is **weight sharing** in a convolutional layer and why does it make CNNs more parameter-efficient than MLPs for image inputs?

**(b)** A CNN is said to be **translation equivariant**. What does this mean? Give a concrete example using the food images in this dataset.

**(c)** A max pooling layer with a 2×2 window and stride 2 is applied to a feature map of size 32×32×64. What is the output size? How many learnable parameters does this pooling layer have, and why?

**Your answer for 1.2:**

(a)

(b)

(c)

## Part 2: MLP Classifier [25 pts]

In this part, you will implement and train a Multilayer Perceptron (MLP) to classify food images. For the MLP, images must be **flattened** into a 1D vector before being fed into the network.

**Requirement:** Your MLP must achieve at least **30% top-1 accuracy** on the test set.

### 2.1 Model Implementation [10 pts]

Implement your MLP by completing the class below. You are free to choose the number of layers and neurons, but your network must satisfy the following:
- At least 2 hidden layers
- ReLU activations
- At least one Dropout layer

The input dimension is 64×64×3 = 12,288.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=64*64*3, num_classes=10):
        super(MLP, self).__init__()
        # TODO: Define your layers here

    def forward(self, x):
        # TODO: Flatten your images before passing them to the network
        pass

mlp_model = MLP().to(device)
print(mlp_model)
total_params = sum(p.numel() for p in mlp_model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

### 2.2 Training [10 pts]

Train your MLP using the training set and evaluate on the validation set each epoch. You must:
- Use `CrossEntropyLoss`
- Train for at least 15 epochs
- Print training loss and validation accuracy per epoch
- Plot training loss and validation accuracy curves

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=20, lr=1e-3):
    # TODO: Define the loss function and optimizer

    train_losses, val_accuracies = [], []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # TODO: Reset gradients from the previous step
            # TODO: Pass images through the model to get predictions
            # TODO: Calculate the loss
            # TODO: Perform backpropagation to compute gradients
            # TODO: Update model weights using the optimizer
            # TODO: Accumulate the loss

        # TODO: Calculate and store the average training loss for this epoch

        model.eval()
        correct, total = 0, 0

        # TODO: Evaluate on the validation set
        # Note: Make sure gradients are not computed during evaluation

        val_acc = 100 * correct / total
        val_accuracies.append(val_acc)
        print(f"Epoch [{epoch+1}/{num_epochs}]  Loss: {avg_loss:.4f}  Val Acc: {val_acc:.2f}%")

    return train_losses, val_accuracies

mlp_train_losses, mlp_val_accs = train_model(mlp_model, train_loader, val_loader, num_epochs=20)

In [ ]:
# TODO: Plot training loss and validation accuracy curves in two side by side subplots.
# Make sure to label your axes and add a title to each plot.

### 2.3 Test Evaluation [5 pts]

Evaluate your trained MLP on the test set. You must meet the 30% threshold to receive full marks.

In [ ]:
def evaluate(model, loader):
    # TODO: Set the model to evaluation mode
    correct, total = 0, 0
    all_preds, all_labels = [], []

    # TODO: Evaluate on the given loader
    # Note: Make sure gradients are not computed during evaluation
    # TODO: Get predictions and accumulate correct predictions
    # TODO: Store predictions and true labels in all_preds and all_labels

    acc = 100 * correct / total
    return acc, all_preds, all_labels

mlp_test_acc, mlp_preds, mlp_labels = evaluate(mlp_model, test_loader)
print(f"MLP Test Accuracy: {mlp_test_acc:.2f}%")
assert mlp_test_acc >= 30.0, f"Accuracy {mlp_test_acc:.2f}% is below the required 30% threshold!"
print("Threshold requirement met.")

## Part 3: CNN Classifier [40 pts]

Now you will implement a CNN. Unlike the MLP, the CNN operates directly on the 2D spatial structure of the image, so do not flatten the input at the beginning.

**Requirement:** Your CNN must achieve at least **50% top-1 accuracy** on the test set.

### 3.1 Model Implementation [15 pts]

Implement your CNN by completing the class below. Your network must satisfy the following:
- At least 3 convolutional layers
- ReLU activations after each conv layer
- At least one max pooling layer
- At least one Dropout layer in the classifier head
- The final layer must output logits for 10 classes

> 💡 **Tip:** If you are struggling to meet the accuracy threshold, consider adding `nn.BatchNorm2d` after your convolutional layers. Batch normalization stabilizes training and often leads to faster convergence and better performance.

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()
        # TODO: Define your convolutional layers here
        self.features = nn.Sequential(
            # YOUR CODE HERE
        )
        # TODO: Define your classifier head here
        self.classifier = nn.Sequential(
            # YOUR CODE HERE
        )

    def forward(self, x):
        x = self.features(x)
        # TODO: Flatten the output of the convolutional layers before passing to the classifier
        return self.classifier(x)

cnn_model = CNN().to(device)
print(cnn_model)
total_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

### 3.2 Training [15 pts]

Train your CNN using the same `train_model` function from Part 2. Plot the training curves.

In [ ]:
# TODO: Train your CNN using the train_model function from Part 2

In [ ]:
# TODO: Plot training loss and validation accuracy curves in two side by side subplots.
# Make sure to label your axes and add a title to each plot.

### 3.3 Test Evaluation [10 pts]

Evaluate your trained CNN on the test set. You must meet the 50% threshold to receive full marks.

In [ ]:
cnn_test_acc, cnn_preds, cnn_labels = # TODO: call the evaluate function on the test set

In [ ]:
print(f"CNN Test Accuracy: {cnn_test_acc:.2f}%")
assert cnn_test_acc >= 50.0, f"Accuracy {cnn_test_acc:.2f}% is below the required 50% threshold!"
print("Threshold requirement met.")

## Part 4: Comparison and Analysis [15 pts]

### 4.1 Side-by-Side Validation Accuracy Comparison [5 pts]

Plot both models' validation accuracy on the same graph.

In [ ]:
# TODO: Plot both models' validation accuracy on the same graph
# Include horizontal dashed lines for the MLP (30%) and CNN (50%) thresholds
# Make sure to label your axes, add a title, and include a legend

print(f"Final MLP Test Accuracy: {mlp_test_acc:.2f}%")
print(f"Final CNN Test Accuracy: {cnn_test_acc:.2f}%")

### 4.2 Discussion [10 pts]

**(a)** If correctly implemented, the CNN should significantly outperform the MLP on this task. Explain why this is expected using the concepts of **locality** and **weight sharing**. Why are these properties particularly useful for food image classification?

**(b)** Look at your training curves for both models. Which model shows signs of overfitting, underfitting, or neither? Suggest one concrete change you could make to improve the weaker model and explain why it would help.

**Your answer for 4.2:**

(a)

(b)